# Phase 1 — Step 1: Dataset Load & Initial EDA
**Project:** Elder Abuse Detection & Legal Assistance — CSE 499A, NSU Spring 2026  
**Dataset:** `data/Elder_abuse_Dataset.csv` (199 cases)

---

## Cell 1 — Libraries Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Libraries loaded ✅')

## Cell 2 — Dataset Load

In [ ]:
df = pd.read_csv('../data/Elder_abuse_Dataset.csv', encoding='utf-8')

print(f'Total rows    : {len(df)}')
print(f'Total columns : {len(df.columns)}')
print()
df.head()

## Cell 3 — Column Names

In [ ]:
print('All Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

## Cell 4 — Data Types & Basic Info

In [ ]:
df.info()

## Cell 5 — Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

if missing_df.empty:
    print('No missing values found ✅')
else:
    print('Columns with missing values:')
    print(missing_df.to_string())

## Cell 6 — Abuse Category (Raw)

In [ ]:
# Find the abuse category column (handle slight name differences)
cat_col = [c for c in df.columns if 'abuse' in c.lower() and 'category' in c.lower()]
cat_col = cat_col[0] if cat_col else 'Abuse Category'

print(f'Column used: "{cat_col}"')
print()
counts = df[cat_col].value_counts()
print(counts.to_string())

## Cell 7 — Abuse Relation (Who is the Abuser?)

In [ ]:
rel_col = [c for c in df.columns if 'relation' in c.lower() or 'abuser' in c.lower()]
rel_col = rel_col[0] if rel_col else 'Abuse Relation'

print(f'Column used: "{rel_col}"')
print()
rel_counts = df[rel_col].value_counts()
print(rel_counts.to_string())

# Trust Blind Spot: family members as abusers
family_keywords = ['son', 'daughter', 'spouse', 'husband', 'wife',
                   'child', 'ছেলে', 'মেয়ে', 'স্বামী', 'স্ত্রী']
family_mask = df[rel_col].str.lower().str.contains(
    '|'.join(family_keywords), na=False
)
trust_pct = family_mask.sum() / len(df) * 100
print(f'\nTrust Blind Spot (family member abusers): {family_mask.sum()} cases = {trust_pct:.1f}%')

## Cell 8 — Gender Distribution

In [ ]:
gen_col = [c for c in df.columns if 'gender' in c.lower()]
gen_col = gen_col[0] if gen_col else 'Gender'

print(f'Column used: "{gen_col}"')
print(df[gen_col].value_counts().to_string())

## Cell 9 — Data Source Type

In [ ]:
type_col = [c for c in df.columns if 'data' in c.lower() and 'type' in c.lower()]
type_col = type_col[0] if type_col else 'Data Type'

print(f'Column used: "{type_col}"')
print(df[type_col].value_counts().to_string())

## Cell 10 — Age Distribution (Raw)

In [ ]:
age_col = [c for c in df.columns if 'age' in c.lower()]
age_col = age_col[0] if age_col else 'Age'

print(f'Column used: "{age_col}"')
print(f'Sample values: {df[age_col].head(10).tolist()}')
print(f'Unique value count: {df[age_col].nunique()}')

# Try numeric conversion
age_numeric = pd.to_numeric(df[age_col], errors='coerce')
print(f'\nNumeric age — valid: {age_numeric.notna().sum()} / {len(df)}')
print(f'Min: {age_numeric.min()}, Max: {age_numeric.max()}, Mean: {age_numeric.mean():.1f}')

## Cell 11 — Location / District Distribution

In [ ]:
loc_col = [c for c in df.columns if 'location' in c.lower() or 'district' in c.lower()]
loc_col = loc_col[0] if loc_col else 'Location'

print(f'Column used: "{loc_col}"')
loc_counts = df[loc_col].value_counts().head(15)
print(loc_counts.to_string())

## Cell 12 — Chart 1: Abuse Category Distribution

In [ ]:
cat_col = [c for c in df.columns if 'abuse' in c.lower() and 'category' in c.lower()][0]
counts = df[cat_col].value_counts()

colors = ['#e74c3c','#e67e22','#f39c12','#27ae60','#2980b9','#8e44ad','#16a085']
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(counts.index, counts.values,
              color=colors[:len(counts)], edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Abuse Category Distribution (Raw)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Abuse Category', fontsize=12)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../data/chart_01_abuse_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → data/chart_01_abuse_category.png ✅')

## Cell 13 — Chart 2: Abuser Relation (Who Abuses?)

In [ ]:
rel_col = [c for c in df.columns if 'relation' in c.lower() or 'abuser' in c.lower()][0]
rel_counts = df[rel_col].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(rel_counts.index[::-1], rel_counts.values[::-1],
               color='#e74c3c', edgecolor='white')

for bar, val in zip(bars, rel_counts.values[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')

ax.set_title('Who Abuses? — Abuser Relation (Top 10)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Cases', fontsize=12)
plt.tight_layout()
plt.savefig('../data/chart_02_abuser_relation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → data/chart_02_abuser_relation.png ✅')

## Cell 14 — Chart 3: Data Source Type

In [ ]:
type_col = [c for c in df.columns if 'data' in c.lower() and 'type' in c.lower()][0]
type_counts = df[type_col].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
colors_pie = ['#2980b9', '#27ae60', '#e74c3c']
wedges, texts, autotexts = ax.pie(
    type_counts.values,
    labels=type_counts.index,
    autopct='%1.1f%%',
    colors=colors_pie[:len(type_counts)],
    startangle=90,
    pctdistance=0.82
)
for t in autotexts:
    t.set_fontsize(12)
    t.set_fontweight('bold')

ax.set_title('Data Source Type Distribution', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../data/chart_03_data_source.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → data/chart_03_data_source.png ✅')

## Cell 15 — Chart 4: Gender Distribution

In [ ]:
gen_col = [c for c in df.columns if 'gender' in c.lower()][0]
gen_counts = df[gen_col].value_counts()

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(gen_counts.index, gen_counts.values,
              color=['#3498db', '#e91e8c', '#95a5a6'][:len(gen_counts)],
              edgecolor='white', width=0.5)

for bar, val in zip(bars, gen_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', fontweight='bold', fontsize=12)

ax.set_title('Victim Gender Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Gender', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
plt.tight_layout()
plt.savefig('../data/chart_04_gender.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → data/chart_04_gender.png ✅')

## Cell 16 — Step 1 Summary

In [ ]:
print('=' * 55)
print('  STEP 1 COMPLETE — Dataset Load & Initial EDA')
print('=' * 55)
print(f'  Total cases        : {len(df)}')
print(f'  Total columns      : {len(df.columns)}')

cat_col = [c for c in df.columns if 'abuse' in c.lower() and 'category' in c.lower()][0]
print(f'  Abuse categories   : {df[cat_col].nunique()} (raw, with typos)')

rel_col = [c for c in df.columns if 'relation' in c.lower() or 'abuser' in c.lower()][0]
family_kw = ['son','daughter','spouse','husband','wife','child','ছেলে','মেয়ে']
family_ct  = df[rel_col].str.lower().str.contains('|'.join(family_kw), na=False).sum()
print(f'  Trust Blind Spot   : {family_ct} cases ({family_ct/len(df)*100:.1f}% family abusers)')

missing_ct = df.isnull().sum().sum()
print(f'  Total missing vals : {missing_ct}')
print()
print('  Charts saved in data/ folder:')
print('  ✅ chart_01_abuse_category.png')
print('  ✅ chart_02_abuser_relation.png')
print('  ✅ chart_03_data_source.png')
print('  ✅ chart_04_gender.png')
print()
print('  NEXT → Step 2: Dataset Cleaning + Train/Test Split')
print('=' * 55)